# Demo 2 — Consume the server you built

Demo 1 consumed *other people's* MCP servers. `mcp.py` builds a small FastMCP server of our own. Start it first, in a terminal:

```sh
uv run python mcp.py
```

Then run the cells below — the **same client pattern** as demo 1, now pointed at our local server.

In [1]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient

In [2]:
load_dotenv()

True

## Connect to the local server and discover its tools

In [3]:
client = MultiServerMCPClient(
    {
        "ciudaty-data": {
            "transport": "streamable_http",
            "url": "http://localhost:8000/mcp",
        }
    }
)
tools = await client.get_tools()
print("tools:", [t.name for t in tools])

tools: ['get_demand_score', 'list_segments']


## Hand them to a plain agent (not RealThor) and ask

In [5]:
provider = os.getenv("LLM_PROVIDER", "openai")
model_kwargs = {"model": "gpt-4o-mini", "temperature": 0}
if provider == "vocareum":
    model_kwargs |= {"base_url": "https://openai.vocareum.com/v1", "api_key": os.getenv("VOCAREUM_API_KEY")}
agent = create_agent(ChatOpenAI(**model_kwargs), tools)

In [6]:
res = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "What is the demand score for a 2-bedroom apartment in the west region?"}]}
)
print(res["messages"][-1].content)

The demand score for a 2-bedroom apartment in the west region is 0.91, with a search volume of 1240.


The tool ran in the `demo_02.py` process, over MCP. The same server is now consumable by any MCP client — including RealThor